# UC-01 Tutorial: Basic Cross-Robot Interaction

This tutorial runs the baseline SEGB simulation (`run_simulation.py`) and explains what to inspect in the web UI.

Goal:
- Generate one interaction graph in memory
- Optionally publish it to SEGB backend
- Verify expected behavior in `/reports` and `/kg-graph`

Prerequisites:
- Open this notebook from inside the SEGB repository.
- To publish data or validate UI views, start backend API at `http://localhost:5000`.
- Python env:
  - `./.venv/bin/python -m pip install -e packages/semantic_log_generator`
- Run the backend preflight cell below before executing the main use-case cell.
- UI routes can be opened on:
  - Production: `http://localhost:8080`
  - Development: `http://localhost:5173`


In [ ]:
# Ensure repository root is available in sys.path
import sys
from pathlib import Path

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'examples').exists() and (candidate / 'packages' / 'semantic_log_generator').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise RuntimeError('Repository root not found. Open this notebook inside the SEGB repository.')

extra_paths = [
    repo_root,
    repo_root / 'packages' / 'semantic_log_generator' / 'src',
    repo_root / 'apps' / 'backend' / 'src',
]
for path in extra_paths:
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

print(f'Using repo root: {repo_root}')


In [ ]:
# Backend preflight (health checks)
import json
import urllib.error
import urllib.request

API_URL = 'http://localhost:5000'
REQUIRE_BACKEND = False


def _get_json(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=5) as response:
        payload = response.read().decode('utf-8')
    return json.loads(payload)


def check_backend_health(api_url: str, *, require: bool) -> None:
    base = api_url.rstrip('/')
    live_url = f"{base}/healthz/live"
    ready_url = f"{base}/healthz/ready"

    try:
        live = _get_json(live_url)
        ready = _get_json(ready_url)
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
        message = f"Backend not reachable at {api_url}: {exc}"
        if require:
            raise RuntimeError(message) from exc
        print('WARNING:', message)
        return

    live_ok = bool(live.get('live'))
    ready_ok = bool(ready.get('ready')) and bool(ready.get('virtuoso'))

    print('healthz/live:', live)
    print('healthz/ready:', ready)

    if not (live_ok and ready_ok):
        message = f"Backend unhealthy at {api_url} (live_ok={live_ok}, ready_ok={ready_ok})"
        if require:
            raise RuntimeError(message)
        print('WARNING:', message)
        return

    print(f"Backend preflight OK -> {api_url}")


check_backend_health(API_URL, require=REQUIRE_BACKEND)


In [ ]:
from examples.simulations.run_simulation import run_basic_simulation

result = run_basic_simulation()
graph = result.graph

print('Use case: UC-01 basic interaction')
print('Human URI:', result.human_uri)
print('Entry SharedEvent URI:', result.entry_shared_event_uri)
print('Speech SharedEvent URI:', result.shared_event_uri)
print('Triples generated:', len(graph))


In [ ]:
# This tutorial does not write local TTL files
print('No local TTL output file is generated in UC-01 notebook.')


## Optional: Publish to Backend

```bash
./.venv/bin/python -m examples.simulations.run_simulation \
  --publish-url http://localhost:5000 \
  --no-print-ttl
```

## What to Verify in the Web UI

Use the same route on prod (`http://localhost:8080`) or dev (`http://localhost:5173`).

1. Open `/session` and set a token if auth is enabled.
2. Open `/reports` and click refresh.
3. Open `/kg-graph` and look for ARI/TIAGo activities linked to one shared event.
4. Open `/query` and query for `oro:ResponseMessage` to verify ARI generated one response.
